In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [34]:
from sklearn.metrics.pairwise import cosine_similarity
import requests, zipfile, io
print("Downloading MovieLens 100K...")
url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall("/tmp/movielens")
ratings = pd.read_csv(
    "/tmp/movielens/ml-100k/u.data",
    sep='\t',
    names=['userId', 'movieId', 'rating', 'timestamp']
)
movies = pd.read_csv(
    "/tmp/movielens/ml-100k/u.item",
    sep='|',
    encoding='latin-1',
    usecols=[0, 1],
    names=['movieId', 'title']
)
ratings = ratings.merge(movies, on='movieId')
ratings.drop(columns='timestamp', inplace=True)

print(ratings.shape)
print(ratings.head())
print(f"\nUnique users  : {ratings['userId'].nunique()}")
print(f"Unique movies : {ratings['movieId'].nunique()}")
print(f"Rating range  : {ratings['rating'].min()} – {ratings['rating'].max()}")

(100000, 4)
   userId  movieId  rating                       title
0     196      242       3                Kolya (1996)
1     186      302       3    L.A. Confidential (1997)
2      22      377       1         Heavyweights (1994)
3     244       51       2  Legends of the Fall (1994)
4     166      346       1         Jackie Brown (1997)

Unique users  : 943
Unique movies : 1682
Rating range  : 1 – 5


In [35]:
movie_matrix = ratings.pivot_table(
    index='userId',
    columns='title',
    values='rating')
movie_matrix = movie_matrix.fillna(0)
print(movie_matrix.head())
print(movie_matrix.shape)

title   'Til There Was You (1997)  1-900 (1994)  101 Dalmatians (1996)  \
userId                                                                   
1                             0.0           0.0                    2.0   
2                             0.0           0.0                    0.0   
3                             0.0           0.0                    0.0   
4                             0.0           0.0                    0.0   
5                             0.0           0.0                    2.0   

title   12 Angry Men (1957)  187 (1997)  2 Days in the Valley (1996)  \
userId                                                                 
1                       5.0         0.0                          0.0   
2                       0.0         0.0                          0.0   
3                       0.0         2.0                          0.0   
4                       0.0         0.0                          0.0   
5                       0.0         0.0          

In [36]:
from sklearn.metrics.pairwise import cosine_similarity
user_similarity = cosine_similarity(movie_matrix)
print(user_similarity.shape)

(943, 943)


In [37]:
similarity_df = pd.DataFrame(
    user_similarity,
    index=movie_matrix.index,
    columns=movie_matrix.index)
print(similarity_df.head())

userId       1         2         3         4         5         6         7    \
userId                                                                         
1       1.000000  0.168937  0.048388  0.064561  0.379670  0.429682  0.443097   
2       0.168937  1.000000  0.113393  0.179694  0.073623  0.242106  0.108604   
3       0.048388  0.113393  1.000000  0.349781  0.021592  0.074018  0.067423   
4       0.064561  0.179694  0.349781  1.000000  0.031804  0.068431  0.091507   
5       0.379670  0.073623  0.021592  0.031804  1.000000  0.238636  0.374733   

userId       8         9         10   ...       934       935       936  \
userId                                ...                                 
1       0.320079  0.078385  0.377733  ...  0.372213  0.119860  0.269860   
2       0.104257  0.162470  0.161273  ...  0.147095  0.310661  0.363328   
3       0.084419  0.062039  0.066217  ...  0.033885  0.043453  0.167140   
4       0.188060  0.101284  0.060859  ...  0.054615  0.036784  0

In [38]:
similar_users = similarity_df[1].sort_values(ascending=False)
print(similar_users.head())

userId
1      1.000000
916    0.566293
864    0.549277
457    0.541364
92     0.539415
Name: 1, dtype: float64


In [46]:
def recommend_movies(user_id):
    similar_users = similarity_df[user_id].sort_values(
        ascending=False)
    similar_users = similar_users.drop(user_id)
    top_user = similar_users.index[0]
    recommended_movies = movie_matrix.loc[top_user]
    recommended_movies = recommended_movies[recommended_movies >= 4]
    recommended_movies = recommended_movies.sort_values(ascending=False)
    return recommended_movies.head(10)

In [40]:
print(recommend_movies(4))

title
Conspiracy Theory (1997)       4.0
Cop Land (1997)                4.0
Edge, The (1997)               4.0
English Patient, The (1996)    4.0
Fly Away Home (1996)           4.0
Full Monty, The (1997)         4.0
Gattaca (1997)                 4.0
Ice Storm, The (1997)          4.0
In & Out (1997)                4.0
Liar Liar (1997)               4.0
Name: 750, dtype: float64


In [47]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(df_filtered,test_size=0.2,random_state=42)

In [48]:
train_matrix = train_data.pivot_table(index='userId',columns='movieId',values='rating').fillna(0)

In [49]:
user_id = train_matrix.index[0]
recommended = recommend_movies(user_id)
relevant = test_data[(test_data['userId'] == user_id)&(test_data['rating'] >= 4)]['movieId'].tolist()
tp = len(set(recommended).intersection(set(relevant)))
precision = tp / len(recommended)
if len(relevant) > 0:
    recall = tp / len(relevant)
else:
    recall = 0
print("Precision:", precision)
print("Recall:", recall)

Precision: 0.0
Recall: 0.0
